In [ ]:
pip install torch
python ejercicio5_scaling.py texto.txt

In [ ]:
import math
import sys
import time
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# Datos
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    return text.replace("\r\n", "\n").replace("\r", "\n")

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def subtext_prefix(ids: torch.Tensor, frac: float) -> torch.Tensor:
    n = len(ids)
    m = max(1000, int(n * frac))  # evita tamaños ridículamente pequeños
    return ids[:m]

class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len
    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)
    def __getitem__(self, idx: int):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y


# -----------------------------
# Mini-Transformer causal
# -----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.drop(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.drop(out)
        return out

class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int, n_head: int, n_layer: int, dropout: float = 0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx):
        B, T = idx.size()
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)


# -----------------------------
# Train / Eval
# -----------------------------
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B*T, V), y.view(B*T))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
        n += 1
    return total / max(1, n)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B*T, V), y.view(B*T))
        total += loss.item()
        n += 1
    return total / max(1, n)


@dataclass
class ScalingConfig:
    seq_len: int = 128
    n_head: int = 4
    dropout: float = 0.1
    batch_size: int = 64
    lr: float = 3e-4
    epochs: int = 3
    device: str = "cpu"
    data_fracs: Tuple[float, ...] = (0.25, 0.50, 1.00)
    models: Tuple[Tuple[str, int, int], ...] = (
        ("small", 2, 128),
        ("medium", 3, 192),
        ("large", 4, 256),
    )  # (label, n_layer, n_embd)


def run_scaling(text_path: str, cfg: ScalingConfig):
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    ids_full = encode(text, stoi)

    device = torch.device(cfg.device)
    vocab_size = len(stoi)

    results: List[Dict[str, Any]] = []

    for frac in cfg.data_fracs:
        ids = subtext_prefix(ids_full, frac)

        # split train/val dentro del subconjunto
        n = len(ids)
        split = int(0.9 * n)
        train_ids = ids[:split]
        val_ids = ids[split:]

        train_ds = CharWindowDataset(train_ids, cfg.seq_len)
        val_ds = CharWindowDataset(val_ids, cfg.seq_len)

        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

        for label, n_layer, n_embd in cfg.models:
            model = MiniTransformerLM(
                vocab_size=vocab_size,
                seq_len=cfg.seq_len,
                n_embd=n_embd,
                n_head=cfg.n_head,
                n_layer=n_layer,
                dropout=cfg.dropout
            ).to(device)

            params = count_params(model)
            optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

            t0 = time.time()
            stable = True
            try:
                for _ in range(cfg.epochs):
                    tr = train_one_epoch(model, train_loader, optim, device)
                    if not math.isfinite(tr):
                        stable = False
                        break
                vl = eval_loss(model, val_loader, device) if stable else float("inf")
            except Exception:
                stable = False
                vl = float("inf")
            t1 = time.time()

            ppl = math.exp(vl) if (stable and vl < 20) else float("inf")
            rec = {
                "data_frac": frac,
                "model": label,
                "layers": n_layer,
                "embd": n_embd,
                "params": params,
                "val_loss": vl,
                "val_ppl": ppl,
                "time_s": round(t1 - t0, 2),
                "stable": stable,
                "train_windows": len(train_ds),
            }
            results.append(rec)

            print(f"data={int(frac*100):3d}% | {label:6s} (L={n_layer},E={n_embd},P={params/1e6:.2f}M) "
                  f"| stable={stable} | val_loss={vl:.4f} | ppl={ppl:.2f} | time={t1-t0:.1f}s")

    # ordenar por val_loss
    results_sorted = sorted(results, key=lambda r: r["val_loss"])
    return results_sorted, results


if __name__ == "__main__":
    #Tamaños de modelo (elige según hardware)

    #CPU (recomendado):
    #Small: n_layer=2, n_embd=128
    #Medium: n_layer=3, n_embd=192
    #Large: n_layer=4, n_embd=256 (opcional si aguanta)

    #GPU (recomendado):
    #Small: 2 x 256
    #Medium: 4 x 256
    #Large: 6 x 384 (si hay margen)

    cfg = ScalingConfig(
        seq_len=128,
        n_head=4,
        dropout=0.1,
        batch_size=64,
        lr=3e-4,
        epochs=3,
        device="cpu",  # cambia a "cuda" si tienes GPU
        data_fracs=(0.25, 0.50, 1.00),
        models=(
            ("small", 2, 128),
            ("medium", 3, 192),
            ("large", 4, 256),
        )
    )

    best_sorted, all_results = run_scaling("texto.txt", cfg)

    print("\n" + "=" * 100)
    print("TOP 10 (menor val_loss):")
    for r in best_sorted[:10]:
        print(r)

    print("\n" + "=" * 100)
    print("Resumen por data_frac:")
    for frac in cfg.data_fracs:
        subset = [r for r in all_results if r["data_frac"] == frac]
        subset = sorted(subset, key=lambda x: x["val_loss"])
        best = subset[0]
        print(f"data={int(frac*100):3d}% -> best_model={best['model']} val_loss={best['val_loss']:.4f} time_s={best['time_s']}")